# Projeto Final: análise das candidaturas da eleição de 2022

**Autores:** Felipe Albanez de Oliveira

## Datasets utilizados
- **Base principal:** consulta_cand_2022_BRASIL.csv
- **Base auxiliar:**  consulta_cand_complementar_2022_BRASIL.csv

### 1. Importando as bibliotecas Pandas e Numpy

In [9]:
# Para facilitar no desenvolvimento, "apelidamos" as bibliotecas no momento da importação
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

### 2. Criando o DataFrame

#### 2.1. Erro na primeira tentativa de criação do DataFrame

In [ ]:
"""
caminho note pessoal
pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_2022_BRASIL.csv"

caminho Trabalho
pasta_arquivo = "/Users/cxxxxxx/OneDrive - Caixa Economica Federal/Área de Trabalho/CAIXA Verso/Projeto Final/Base de dados/consulta_cand_2022_BRASIL.csv"

df_original = pd.read_csv(pasta_arquivo)
df_original.head()

erro gerado:
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc7 in position 838: invalid continuation byte

"""

#### 2.2. Segunda tentativa - modo "a brasileira"

In [ ]:
""""
sep=';': Os dados usam ponto e vírgula para separar as colunas
decimal=",": números decimais usam vírgula
encoding='latin-1': Evita erros de leitura com acentos e caracteres da língua portuguesa.
"""
# substitua o caminho abaixo pela pasta onde está salvo a base de dados .csv
# não esquecer que o nome do arquivo e a extensão deve estar inclusos

# caminho note pessoal
pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_2022_BRASIL.csv"

# caminho Trabalho
# pasta_arquivo = "/Users/cxxxxxx/OneDrive - Caixa Economica Federal/Área de Trabalho/CAIXA Verso/Projeto Final/Base de dados/consulta_cand_2022_BRASIL.csv"

# cria o dataframe lendo o arquivo .csv
df_original = pd.read_csv(pasta_arquivo, sep=';', decimal=',',encoding='latin-1')

In [ ]:
# vizualisa as 5 primeiras linhas do dataframe
df_original.head()

### 3. Conhecendo o DataFrame

#### 3.1. Dimensões e Estrutura do DataFrame

In [ ]:
# shape: informa o número de linhas e colunas
print(f'Shape: {df_original.shape}')
print(f'Quantidade de linhas: {df_original.shape[0]}')
print(f'Quantidade de colunas: {df_original.shape[1]}')

In [ ]:
# info: informa os tipos, faltantes, memória
df_original.info()        

In [ ]:
# describle aula
display(df_original.describe().round(2))
display(df_original.describe(include='object'))

#### 3.2 Avaliando as categorizaçãos

In [ ]:
# Mostra os nomes das colunas.
df_original.columns

In [ ]:
# Avaliação de algumas colunas que serão utilizadas
print(df_original['DS_GENERO'].value_counts(dropna=False))
print()
print(df_original['DS_COR_RACA'].value_counts(dropna=False))
print()
print(df_original['DS_GRAU_INSTRUCAO'].value_counts(dropna=False))

#### 3.3. Verificação de Faltantes

#### 3.3.1. Faltantes puros

In [ ]:
# cria um df chamado Faltantes
# são criadas tres colunas para o df 
# isna.sum: percorre a coluna e retorna a soma dos faltantes
# isna.mean: percorre a coluna e retorna a média dos faltantes em relação ao número total de itens na coluna
# nunique: conta quantos valores diferentes cada coluna tem
faltantes = pd.DataFrame({
    'Faltantes': df_original.isna().sum(),
    '%': (df_original.isna().mean() * 100).round(2),
    'distintos': df_original.nunique(),
})

# filta somente as colunas que não possui faltantes
# sort e ascending: ordena pela coluna faltante do maior para o menor
faltantes[faltantes['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

Devido a baixíssima quantidades de faltantes, não é necessário a exclusão da coluna

#### 3.3.2. Outros tipos de faltantes
Analisando a tabela, verificamos que existe células preenchidas com texto que podem ser considerados 'faltantes'

In [ ]:
nulos = pd.DataFrame({
    '#NULO': (df_original == '#NULO').sum(),
    '%': ((df_original == '#NULO').mean() * 100).round(2),
})

nulos[nulos['#NULO'] > 0].sort_values('#NULO', ascending=False)

In [ ]:
naodivulgavel = pd.DataFrame({
    'NÃO DIVULGÁVEL': (df_original == 'NÃO DIVULGÁVEL').sum(),
    '%': ((df_original == 'NÃO DIVULGÁVEL').mean() * 100).round(2),
})

naodivulgavel[naodivulgavel['NÃO DIVULGÁVEL'] > 0].sort_values('NÃO DIVULGÁVEL', ascending=False)

#### 3.3.3. Substituindo por NaN
#NULO e NÃO DIVULGAVEL são strings, portanto é necessário transformar em NaN

In [ ]:
df_original = df_original.replace('#NULO', np.nan)
df_original = df_original.replace('NÃO DIVULGÁVEL', np.nan)

#### 3.3.4. Contagem de Faltantes após o tratamento

In [ ]:
faltantes_atualizado = pd.DataFrame({
    'Faltantes_atualizado': df_original.isna().sum(),
    '%': (df_original.isna().mean() * 100).round(2),
})

# filta somente as colunas que não possui faltantes
# sort e ascending: ordena pela coluna faltante do maior para o menor
faltantes_atualizado[faltantes_atualizado['Faltantes_atualizado'] > 0].sort_values('Faltantes_atualizado', ascending=False)

#### 3.3. Verificação de Duplicados

In [25]:
# verifica se existe linhas com todos os dados duplicados
print('Linhas totalmente duplicadas:', df_original.duplicated().sum())

# o id deve ser sempre único, portanto é necessário analisar possível duplicidade
print('Candidatos duplicados:', df_original['SQ_CANDIDATO'].duplicated().sum())

# df filtrado somente com "id" duplicados
df_original[df_original.duplicated(keep=False)].sort_values('SQ_CANDIDATO').head(6)

Linhas totalmente duplicadas: 0
Candidatos duplicados: 52


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,...,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO


#### 3.4. Categorias inconsistentes

In [ ]:
# retorna todas as colunas do df
df_prf_original.columns

Index(['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km',
       'municipio', 'causa_acidente', 'tipo_acidente',
       'classificacao_acidente', 'fase_dia', 'sentido_via',
       'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo',
       'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos',
       'ignorados', 'feridos', 'veiculos', 'latitude', 'longitude', 'regional',
       'delegacia', 'uop'],
      dtype='object')

In [ ]:
# verificação dos dias da semana
df_prf_original['dia_semana'].value_counts()

In [ ]:
# verificação da escrita dos estados
df_prf_original['uf'].value_counts()

In [ ]:
df_prf_original['classificacao_acidente'].value_counts()

### 3.5. Valores inválidos

### 4. Definindo quais colunas do Dataframe serão utililizadas 

In [ ]:
# imprimi as colunas do df
print(df_prf_original.columns)

In [ ]:
# quais valores cada coluna possui
# unique: retorna uma lista com os valores únicos daquela coluna
# valeu_counts

uf = df_prf_original['uf'].unique()
print("uf:", uf)

df_prf_original['uf'].value_counts()



In [ ]:
# cria uma lista com os nomes das colunas

# colunas_selecionadas = []